# The Term Structure of the Variance Risk Premium
## Evidence Across 1M, 3M, and 6M Horizons, 2014–2025

### Research Question

Where along the 1M–3M–6M SPX volatility curve is the variance risk premium concentrated, and how stable is that term structure across market conditions?

This project addresses two related questions:

1. Is the ex-post variance risk premium systematically larger at longer maturities?
2. Does the slope of the VRP term structure vary with the level of market volatility?

## 1. Data and Sample

The analysis uses daily observations for:

- S&P 500 Index (SPX)
- Cboe Volatility Index (VIX, approximately 1-month horizon)
- Cboe 3-Month Volatility Index (VIX3M)
- Cboe 6-Month Volatility Index (VIX6M)

The source files were originally provided at 1-minute frequency and were aggregated to daily closes using the final available observation at or before 4:00 PM Eastern Time.

The common four-series sample begins on June 4, 2014 because the available VIX6M intraday history starts on that date.

Because the 6-month ex-post realized variance requires approximately 184 subsequent calendar days, the fully observable three-maturity VRP sample ends before the January 2026 raw-data endpoint.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:.6f}")

DATA = Path("../data/processed")
FIGURES = Path("../outputs/figures")

df = pd.read_csv(
    DATA / "vrp_term_structure.csv",
    parse_dates=["date"]
)

df.head()

## 2. Data Validation

The intraday-to-daily aggregation was validated in three ways:

1. SPX, VIX, VIX3M and VIX6M overwhelmingly share the same normal-session closing timestamp.
2. Early-close days were inspected manually; apparent late SPX timestamps were stale vendor bars carrying an unchanged closing value rather than economically new observations.
3. Selected SPX and VIX daily closes were compared with independently published daily closes. Differences were negligible.

In [ ]:
df[["date", "spx", "vix", "vix3m", "vix6m"]].describe()

## 3. Methodology

The analysis defines the ex-post variance risk premium as:

\[
VRP_{t,h} = IV_{t,h} - RV_{t,t+h}
\]

where:

- \(IV_{t,h}\) is the annualized implied variance observed at date \(t\)
- \(RV_{t,t+h}\) is the annualized realized SPX variance over the subsequent horizon

The implied variance proxies are:

\[
IV_{t,1M} = \left(\frac{VIX_t}{100}\right)^2
\]

\[
IV_{t,3M} = \left(\frac{VIX3M_t}{100}\right)^2
\]

\[
IV_{t,6M} = \left(\frac{VIX6M_t}{100}\right)^2
\]

Forward realized variance is computed from squared daily SPX log returns and annualized using 252 trading days.

The target horizons are approximately:

- 1M: 30 calendar days
- 3M: 93 calendar days
- 6M: 184 calendar days

A positive VRP therefore means implied variance exceeded the variance subsequently realized by the SPX.

In [ ]:
main = df[["vrp_1m", "vrp_3m", "vrp_6m"]].describe().T
main

## 4. Main Results

The average VRP increases with maturity:

- 1M: approximately 0.0066
- 3M: approximately 0.0127
- 6M: approximately 0.0169

The median VRP is also positive at all three maturities.

This suggests that the ex-post implied-minus-realized variance gap is larger, on average, toward the longer end of the volatility curve.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(FIGURES / "01_average_vrp_term_structure.png")))

## 5. Term-Structure Stability

The average ordering of VRP is upward sloping, but the shape is not constant through time.

In the daily sample:

- \(VRP_{3M} > VRP_{1M}\) approximately 77.7% of the time
- \(VRP_{6M} > VRP_{3M}\) approximately 73.8% of the time
- \(VRP_{6M} > VRP_{1M}\) approximately 73.6% of the time

The strongest statistical evidence appears between the endpoints of the curve rather than between adjacent maturities.

In [ ]:
display(Image(filename=str(FIGURES / "03_vrp_slope_over_time.png")))

## 6. Volatility-Regime Analysis

The term structure changes materially with the contemporaneous level of VIX.

In low-volatility regimes, the 6M–1M VRP slope is close to zero or slightly negative.

As VIX rises, the slope becomes increasingly positive.

In the highest VIX quintile, the average 6M–1M spread is approximately 0.033, compared with approximately -0.006 in the lowest quintile.

A continuous daily regression of the 6M–1M VRP slope on VIX gives:

\[
\beta_{VIX} \approx 0.00193
\]

with a HAC-adjusted p-value of approximately 0.009.

This relationship is descriptive rather than causal.

In [ ]:
display(Image(filename=str(FIGURES / "02_vrp_by_vix_regime.png")))

## 7. Robustness Checks

### HAC / Newey-West inference

Because forward realized-variance windows overlap heavily, daily VRP observations are strongly serially dependent.

HAC/Newey-West standard errors are therefore used.

The main long-short result is:

\[
VRP_{6M} - VRP_{1M} \approx 0.01075
\]

with a daily HAC p-value of approximately 0.038.

### Month-end robustness

Using only month-end observations reduces the amount of overlapping information.

The month-end estimate is:

\[
VRP_{6M} - VRP_{1M} \approx 0.01057
\]

with a HAC p-value of approximately 0.042.

The magnitude is therefore almost unchanged despite reducing the sample from roughly 2,800 daily observations to 134 month-end observations.

The volatility-regime relationship is also directionally preserved in the month-end sample. The VIX coefficient remains positive, although statistical precision declines (p ≈ 0.064).

In [ ]:
hac = pd.read_csv(DATA / "hac_tests.csv")
hac

## 8. Limitations

This V1 analysis has several important limitations.

First, the common sample begins in June 2014 because the currently available VIX6M intraday data do not extend further back. A future version should recover longer official histories and extend the analysis toward 2010.

Second, VIX, VIX3M and VIX6M are used as implied-variance proxies. The resulting quantities are not direct tradable variance-swap P&Ls.

Third, realized variance is constructed from daily close-to-close SPX returns rather than higher-frequency realized-variance estimators.

Fourth, regime results describe conditional associations and should not be interpreted causally.

Finally, major market shocks such as the COVID-19 episode have a substantial influence on ex-post realized variance, particularly at longer horizons.

## 9. Conclusion

The V1 evidence supports three main conclusions.

First, the ex-post variance risk premium is upward sloping on average across the 1M, 3M and 6M horizons.

Second, the strongest statistical evidence appears between the endpoints of the curve. The average 6M–1M spread is positive and remains statistically significant after HAC correction in both the daily and month-end samples.

Third, the slope is state-dependent. The curve is relatively flat in low-volatility environments and becomes substantially steeper when VIX is elevated.

Overall, the evidence suggests that the VRP term structure is not a fixed feature of the market. Its shape varies meaningfully with market conditions, with the long end becoming especially important during periods of elevated implied volatility.